In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tentukan opsyen

<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Anda boleh menggunakan opsyen untuk menyesuaikan primitif Estimator dan Sampler. Bahagian ini memberi tumpuan kepada cara menentukan opsyen primitif Qiskit Runtime. Walaupun antara muka kaedah `run()` primitif adalah sama merentasi semua pelaksanaan, opsyen mereka tidak sama. Rujuk rujukan API yang bersesuaian untuk maklumat tentang opsyen [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) dan [`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html).

Nota tentang menentukan opsyen dalam primitif:

- `SamplerV2` dan `EstimatorV2` mempunyai kelas opsyen yang berasingan. Anda boleh melihat opsyen yang tersedia dan mengemas kini nilai opsyen semasa atau selepas pemulaan primitif.
- Gunakan kaedah `update()` untuk menggunakan perubahan pada atribut `options`.
- Jika anda tidak menentukan nilai untuk sesuatu opsyen, ia diberi nilai khas `Unset` dan lalai pelayan digunakan.
- Atribut `options` adalah jenis Python `dataclass`. Anda boleh menggunakan kaedah `asdict` terbina dalam untuk menukarnya kepada kamus.

<span id="pass-options"></span>
## Tetapkan opsyen primitif
Anda boleh menetapkan opsyen semasa memulakan primitif, selepas memulakan primitif, atau dalam kaedah `run()`. Lihat bahagian [peraturan keutamaan](runtime-options-overview#options-precedence) untuk memahami apa yang berlaku apabila opsyen yang sama ditentukan di beberapa tempat.

### Pemulaan primitif
Anda boleh menghantar contoh kelas opsyen atau kamus semasa memulakan primitif, yang kemudiannya membuat salinan opsyen tersebut. Oleh itu, mengubah kamus asal atau contoh opsyen tidak mempengaruhi opsyen yang dimiliki oleh primitif.

#### Kelas opsyen
Semasa mencipta contoh kelas `EstimatorV2` atau `SamplerV2`, anda boleh menghantar contoh kelas opsyen. Opsyen tersebut kemudiannya akan digunakan apabila anda menggunakan `run()` untuk melakukan pengiraan. Tentukan opsyen dalam format ini: `options.option.sub-option.sub-sub-option = choice`. Contohnya: `options.dynamical_decoupling.enable = True`

Contoh:

`SamplerV2` dan `EstimatorV2` mempunyai kelas opsyen yang berasingan ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) dan [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Kamus
Anda boleh menentukan opsyen sebagai kamus semasa memulakan primitif.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Kemas kini opsyen selepas pemulaan
Anda boleh menentukan opsyen dalam format ini: `primitive.options.option.sub-option.sub-sub-option = choice` untuk memanfaatkan auto-lengkap, atau gunakan kaedah `update()` untuk membuat kemas kini pukal.

Kelas opsyen `SamplerV2` dan `EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) dan [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) tidak perlu dibuat contoh jika anda menetapkan opsyen selepas memulakan primitif.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Kaedah Run()
Satu-satunya nilai yang boleh anda hantar kepada `run()` ialah nilai yang ditakrifkan dalam antara muka. Iaitu, `shots` untuk Sampler dan `precision` untuk Estimator. Ini mengatasi mana-mana nilai yang ditetapkan untuk `default_shots` atau `default_precision` untuk jalankan semasa.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### Kes khas
#### Tahap ketahanan (Estimator sahaja)
Tahap ketahanan sebenarnya bukan opsyen yang secara langsung mempengaruhi pertanyaan primitif, tetapi menentukan set asas opsyen yang terkurasi untuk dibina. Secara umum, tahap 0 mematikan semua pengurangan ralat, tahap 1 menghidupkan opsyen untuk pengurangan ralat pengukuran, dan tahap 2 menghidupkan opsyen untuk pengurangan ralat Gate dan pengukuran.

Mana-mana opsyen yang anda tentukan secara manual sebagai tambahan kepada tahap ketahanan digunakan di atas set asas opsyen yang ditakrifkan oleh tahap ketahanan. Oleh itu, pada prinsipnya, anda boleh menetapkan tahap ketahanan kepada 1, tetapi kemudian mematikan pengurangan ralat pengukuran, walaupun ini tidak disyorkan.

Dalam contoh berikut, menetapkan tahap ketahanan kepada 0 pada mulanya mematikan `zne_mitigation`, tetapi `estimator.options.resilience.zne_mitigation = True` mengatasi persediaan yang berkaitan daripada `estimator.options.resilience_level = 0`.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### Shot (Sampler sahaja)
Kaedah `SamplerV2.run` menerima dua argumen: senarai PUB, yang masing-masing boleh menentukan nilai shot khusus PUB, dan argumen kata kunci shots. Nilai shot ini adalah sebahagian daripada antara muka pelaksanaan Sampler, dan bebas daripada opsyen Sampler Runtime. Mereka mengambil keutamaan berbanding mana-mana nilai yang ditentukan sebagai opsyen untuk mematuhi abstraksi Sampler.

Namun, jika `shots` tidak ditentukan oleh mana-mana PUB atau dalam argumen kata kunci jalankan (atau jika semuanya `None`), maka nilai shots daripada opsyen digunakan, terutamanya `default_shots`.

Ringkasnya, ini adalah susunan keutamaan untuk menentukan shots dalam Sampler, untuk mana-mana PUB tertentu:

1. Jika PUB menentukan shots, gunakan nilai itu.
2. Jika argumen kata kunci `shots` ditentukan dalam `run`, gunakan nilai itu.
3. Jika `num_randomizations` dan `shots_per_randomization` ditentukan sebagai opsyen `twirling`, shots adalah hasil darab nilai-nilai tersebut.
3. Jika `sampler.options.default_shots` ditentukan, gunakan nilai itu.

Oleh itu, jika shots ditentukan di semua tempat yang mungkin, yang mempunyai keutamaan tertinggi (shots yang ditentukan dalam PUB) digunakan.

#### Ketepatan (Estimator sahaja)
Ketepatan adalah analog kepada shots, seperti yang diterangkan dalam bahagian sebelumnya, kecuali bahawa opsyen Estimator mengandungi kedua-dua `default_shots` dan `default_precision`. Selain itu, kerana gate-twirling didayakan secara lalai, hasil darab `num_randomizations` dan `shots_per_randomization` mengambil keutamaan berbanding kedua-dua opsyen tersebut.

Secara khusus, untuk mana-mana PUB Estimator tertentu:

1. Jika PUB menentukan ketepatan, gunakan nilai itu.
2. Jika argumen kata kunci ketepatan ditentukan dalam `run`, gunakan nilai itu.
2. Jika `num_randomizations` dan `shots_per_randomization` ditentukan sebagai [opsyen `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (didayakan secara lalai), gunakan hasil darab mereka untuk mengawal jumlah data.
3. Jika `estimator.options.default_shots` ditentukan, gunakan nilai itu untuk mengawal jumlah data.
4. Jika `estimator.options.default_precision` ditentukan, gunakan nilai itu.

Contohnya, jika ketepatan ditentukan di kesemua empat tempat, yang mempunyai keutamaan tertinggi (ketepatan yang ditentukan dalam PUB) digunakan.

> **Note:** Ketepatan berskala songsang dengan penggunaan. Iaitu, semakin rendah ketepatan, semakin banyak masa QPU yang diperlukan untuk menjalankannya.

## Opsyen yang biasa digunakan
Terdapat banyak opsyen yang tersedia, tetapi berikut adalah yang paling biasa digunakan:

<span id="shots"></span>
### Shot
Untuk sesetengah algoritma, menetapkan bilangan shot tertentu adalah bahagian teras rutin mereka. Shot (atau ketepatan) boleh ditentukan di beberapa tempat. Ia diprioritaskan seperti berikut:

Untuk mana-mana PUB Sampler:

1. Shot bernilai integer yang terkandung dalam PUB
2. Nilai `run(...,shots=val)`
3. Nilai `options.default_shots`

Untuk mana-mana PUB Estimator:

1. Ketepatan bernilai titik apung yang terkandung dalam PUB
2. Nilai `run(...,precision=val)`
3. Nilai `options.default_shots`
4. Nilai `options.default_precision`

Contoh:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### Masa pelaksanaan maksimum
Masa pelaksanaan maksimum (`max_execution_time`) mengehadkan berapa lama sesuatu kerja boleh berjalan. Jika kerja melebihi had masa ini, ia dibatalkan secara paksa. Nilai ini digunakan untuk kerja tunggal, sama ada dijalankan dalam mod kerja, Session, atau batch.

Nilai ditetapkan dalam saat, berdasarkan masa kuantum (bukan masa jam dinding), iaitu jumlah masa yang QPU didedikasikan untuk memproses kerja anda. Ia diabaikan apabila menggunakan mod ujian tempatan kerana mod tersebut tidak menggunakan masa kuantum.